# 01 - EDA and Graph Construction
Validates synthetic data quality and confirms graph signal strength for downstream GNN training.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style='whitegrid')
print(f'Using project root: {PROJECT_ROOT}')

In [ ]:
data_dir = Path('data')
borrowers = pd.read_parquet(data_dir / 'borrowers.parquet')
loans = pd.read_parquet(data_dir / 'loans.parquet')
repayments = pd.read_parquet(data_dir / 'repayments.parquet')
groups = pd.read_parquet(data_dir / 'groups.parquet')
edges = pd.read_parquet(data_dir / 'edges.parquet')
cdr = pd.read_parquet(data_dir / 'cdr.parquet')
mobile_events = pd.read_parquet(data_dir / 'mobile_events.parquet')
labels = pd.read_parquet(data_dir / 'labels.parquet')

print('rows:', {
    'borrowers': len(borrowers),
    'loans': len(loans),
    'repayments': len(repayments),
    'groups': len(groups),
    'edges': len(edges),
    'cdr': len(cdr),
    'mobile_events': len(mobile_events),
    'labels': len(labels),
})

In [ ]:
joined = (
    borrowers[['borrower_id', 'group_id', 'CoDi_wallet_flag']]
    .merge(labels, on='borrower_id', how='left')
    .merge(loans[['loan_id', 'borrower_id']], on='borrower_id', how='left')
    .merge(repayments[['loan_id', 'repayment_latency_days']], on='loan_id', how='left')
)

g = nx.Graph()
g.add_nodes_from(borrowers['borrower_id'].astype(int).tolist())
g.add_weighted_edges_from(edges[['src_id', 'dst_id', 'tie_strength']].itertuples(index=False, name=None))

degree_map = dict(g.degree())
clustering_map = nx.clustering(g, weight='weight')
pagerank_map = nx.pagerank(g, weight='weight', alpha=0.85, max_iter=100)
communities = nx.community.louvain_communities(g, weight='weight', seed=42)

print('Graph nodes:', g.number_of_nodes())
print('Graph edges:', g.number_of_edges())
print('Avg degree:', np.mean(list(degree_map.values())))
print('Avg clustering:', np.mean(list(clustering_map.values())))
print('Detected communities:', len(communities))

In [ ]:
stats = borrowers[['borrower_id', 'group_id', 'CoDi_wallet_flag']].merge(labels, on='borrower_id', how='left').copy()
stats['degree'] = stats['borrower_id'].map(degree_map).fillna(0.0)
stats['clustering'] = stats['borrower_id'].map(clustering_map).fillna(0.0)
stats['pagerank'] = stats['borrower_id'].map(pagerank_map).fillna(0.0)
stats['centrality_decile'] = pd.qcut(stats['pagerank'].rank(method='first'), 10, labels=False)

cohesion_map = groups.set_index('group_id')['cohesion_score']
stats['group_cohesion'] = stats['group_id'].map(cohesion_map)

latency_by_borrower = joined.groupby('borrower_id', as_index=False)['repayment_latency_days'].mean()
stats = stats.merge(latency_by_borrower, on='borrower_id', how='left')

def compute_neighbor_default_rate(graph, labels_df):
    y_map = labels_df.set_index('borrower_id')['default_flag'].to_dict()
    rates = {}
    for node in graph.nodes():
        nbrs = list(graph.neighbors(node))
        if not nbrs:
            rates[node] = float(np.mean(list(y_map.values())))
        else:
            rates[node] = float(np.mean([y_map[n] for n in nbrs]))
    return rates

nbr_rate = compute_neighbor_default_rate(g, labels)
stats['neighbor_default_rate'] = stats['borrower_id'].map(nbr_rate).fillna(stats['default_flag'].mean())
corr = float(stats[['neighbor_default_rate', 'default_flag']].corr().iloc[0, 1])
print('Correlation(neighbor_default_rate, default_flag)=', round(corr, 4))
stats.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

d1 = stats.groupby('centrality_decile', as_index=False)['default_flag'].mean()
sns.barplot(data=d1, x='centrality_decile', y='default_flag', ax=axes[0], color='#3a7')
axes[0].set_title('Default Rate by PageRank Decile')
axes[0].set_ylabel('Default rate')

d2 = stats.groupby(pd.qcut(stats['group_cohesion'], 10, duplicates='drop'), as_index=False)['default_flag'].mean()
sns.lineplot(data=d2, x=d2.index, y='default_flag', marker='o', ax=axes[1], color='#c65')
axes[1].set_title('Default Rate by Group Cohesion Decile')
axes[1].set_xlabel('Cohesion decile index')

d3 = stats.groupby('CoDi_wallet_flag', as_index=False)['repayment_latency_days'].mean()
sns.barplot(data=d3, x='CoDi_wallet_flag', y='repayment_latency_days', ax=axes[2], palette='viridis')
axes[2].set_title('Repayment Latency by CoDi Wallet Presence')
axes[2].set_xlabel('CoDi wallet flag')

plt.tight_layout()
plt.show()

## Data Quality Notes
- Verify class balance remains within 15-20% default range from generation target.
- Confirm graph signal threshold: correlation between neighborhood default rate and individual default should exceed 0.15.
- Inspect outliers in repayment latency and tie strength before model training.